#OPTIMIZATION

###1. Import Library Pandas, Dask, Polar and  Create a Measurement Performance

In [1]:
import pandas as pd
import dask.dataframe as dd
import polars as pl
import time
import psutil
import os

FILE_NAME = "theedge_100k_master.csv"

def measure_performance(func, *args):
    """A wrapper function to measure Time, CPU, Memory, and Throughput."""
    process = psutil.Process(os.getpid())

    # Baseline measurements
    start_time = time.time()
    start_mem = process.memory_info().rss
    psutil.cpu_percent(interval=None) # Start CPU tracking

    # Run the processing function
    result_df, record_count = func(*args)

    # Final measurements
    end_time = time.time()
    end_mem = process.memory_info().rss
    cpu_util = psutil.cpu_percent(interval=None)

    # Calculate metrics
    total_time = end_time - start_time
    peak_memory_mb = max(0, (end_mem - start_mem)) / (1024 * 1024)
    throughput = record_count / total_time if total_time > 0 else 0

    return {
        "Total Processing Time (s)": round(total_time, 4),
        "Average CPU Util (%)": round(cpu_util, 2),
        "Peak Memory (MB)": round(peak_memory_mb, 2),
        "Throughput (recs/sec)": round(throughput, 2)
    }, result_df

###2. BASELINE PIPELINE (STANDARD PANDAS)

In [ ]:
def clean_data_baseline(filepath):
    print("Running Baseline (Pandas) Pipeline...")
    df = pd.read_csv(filepath)

    # Cleaning
    df['author'] = df['author'].fillna("Unspecified")
    df['summary'] = df['summary'].fillna("No summary available")
    df['sub-category'] = df['sub-category'].fillna("No sub-category")
    df['updated date'] = df['updated date'].fillna("No updated date")

    # NEW: Remove weird encoding symbols (Mojibake)
    df['title'] = df['title'].str.replace(r'[^\x00-\x7F]+', '', regex=True)
    df['summary'] = df['summary'].str.replace(r'[^\x00-\x7F]+', '', regex=True)
    df = df.dropna(subset=['title', 'link'])

    df['category'] = df['category'].astype(str).str.title()
    df['title'] = df['title'].astype(str).str.strip()
    df = df.drop_duplicates(subset=['title'])

    return df, len(df)

### 3. OPTIMIZED PIPELINE 1 (DASK)


In [ ]:
def clean_data_dask(filepath):
    print("Running Optimized 1 (Dask) Pipeline...")
    df = dd.read_csv(filepath, blocksize="16MB")

    # Cleaning
    df['author'] = df['author'].fillna("Unspecified")
    df['summary'] = df['summary'].fillna("No summary available")
    df['sub-category'] = df['sub-category'].fillna("Unspecified")
    df['updated date'] = df['updated date'].fillna("Unspecified")

    # NEW: Remove weird encoding symbols (Mojibake)
    df['title'] = df['title'].str.replace(r'[^\x00-\x7F]+', '', regex=True)
    df['summary'] = df['summary'].str.replace(r'[^\x00-\x7F]+', '', regex=True)

    df = df.dropna(subset=['title', 'link'])

    df['category'] = df['category'].astype(str).str.title()
    df['title'] = df['title'].astype(str).str.strip()
    df = df.drop_duplicates(subset=['title'])

    # Execute the lazy graph
    result_df = df.compute()
    return result_df, len(result_df)

###4. OPTIMIZED PIPELINE 2 (POLARS)

In [ ]:
def clean_data_polars(filepath):
    print("Running Optimized 2 (Polars) Pipeline...")

    # scan_csv enables lazy evaluation for maximum speed and memory efficiency
    lazy_df = pl.scan_csv(filepath)

    # Polars syntax is slightly different but highly optimized
    cleaned_lazy_df = (
        lazy_df
        # Step 1: Fill null values for all requested columns
        .with_columns([
            pl.col("author").fill_null("Unknown"),
            pl.col("summary").fill_null("No summary available"),
            pl.col("sub-category").fill_null("Unspecified"),
            pl.col("updated date").fill_null("Unspecified"),
        ])
        .drop_nulls(subset=["title", "link"])

        # Step 2: Format and clean text (Chaining methods together to avoid duplicates!)
        .with_columns([
            pl.col("category").str.to_titlecase(),

            pl.col("title").str.strip_chars().str.replace_all(r'[^\x00-\x7F]+', ''),

            # Clean the 'summary' column from weird symbols
            pl.col("summary").str.replace_all(r'[^\x00-\x7F]+', '')
        ])
        # Step 3: Remove duplicates
        .unique(subset=["title"], keep="first")
    )

    # .collect() executes the lazy operations on all CPU cores simultaneously
    result_df = cleaned_lazy_df.collect()

    return result_df, len(result_df)

###5. EXECUTE AND COMPARE ALL THREE

In [ ]:
if __name__ == "__main__":
    if not os.path.exists(FILE_NAME):
        print(f"Error: {FILE_NAME} not found. Please place it in the same directory.")
    else:
        # Run all three tests
        baseline_metrics, clean_pandas_df = measure_performance(clean_data_baseline, FILE_NAME)
        dask_metrics, clean_dask_df = measure_performance(clean_data_dask, FILE_NAME)
        polars_metrics, clean_polars_df = measure_performance(clean_data_polars, FILE_NAME)

        # Save the Polars version as our final cleaned dataset (since it's usually the fastest)
        clean_polars_df.write_csv("cleaned_data_final.csv")
        print("\n💾 Saved finalized dataset to 'cleaned_data_final.csv'")

        # Display Results for Deliverable 4
        print("\n" + "="*70)
        print("📊 PERFORMANCE COMPARISON RESULTS")
        print("="*70)

        results_df = pd.DataFrame(
            [baseline_metrics, dask_metrics, polars_metrics],
            index=["Baseline (Pandas)", "Optimized (Dask)", "Optimized (Polars)"]
        )
        print(results_df.to_markdown())
        results_df.to_csv("performance_comparison.csv")